# LIBS Instrument Calibration Report – Multi-Material

**Instrument:** <INSTRUMENT_NAME>

This report summarises the multi-material instrument profile calibration process.
Plasma parameters were estimated independently for each reference material and then
averaged per zone to produce the final instrument profile.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
print(f"Report generated on: {time.strftime('%d-%m-%Y %H:%M:%S')}")

In [ ]:
# --- CONFIGURATION (injected by LIBSDataCurator) ---
instrument_name           = "<INSTRUMENT_NAME>"
averaged_zones_csv_path   = "<AVERAGED_ZONES_CSV_PATH>"
per_material_zones_dir    = "<PER_MATERIAL_ZONES_CSV_DIR>"
material_names            = <MATERIAL_NAMES_LIST>
num_materials             = <NUM_MATERIALS_PROCESSED>
rmse                      = <RMSE>
rsquare_score             = <RSQUARE_SCORE>

# Baseline correction parameters
baseline_lambda           = <LAMBDA>
baseline_p                = <P>
baseline_max_iter         = <MAX_ITERATIONS>

print(f"Instrument       : {instrument_name}")
print(f"Materials        : {num_materials}")
print(f"Avg R\u00b2          : {rsquare_score:.4f}")
print(f"Avg RMSE         : {rmse:.4f}")
print(f"Baseline \u03bb/p/iter: {baseline_lambda} / {baseline_p} / {baseline_max_iter}")

## 1. Per-Material Best Zone Parameters
The table below lists the best-fit plasma parameters estimated for each reference material.

In [ ]:
per_mat_rows = []
for mat_name in material_names:
    csv_path = os.path.join(per_material_zones_dir, f"{mat_name}_best_zones.csv")
    if not os.path.exists(csv_path):
        print(f"  Warning: zones CSV not found for material '{mat_name}'")
        continue
    mat_df = pd.read_csv(csv_path, header=0)
    for zone_idx, row in mat_df.iterrows():
        per_mat_rows.append({
            'Material': mat_name,
            'Zone':     zone_idx + 1,
            'Te (eV)':  row['Te'],
            'Ne (cm\u207b\u00b3)': row['Ne'],
            'Weight':   row['Weight']
        })

if per_mat_rows:
    summary_df = pd.DataFrame(per_mat_rows)
    print(summary_df.to_string(index=False))
else:
    print("No per-material zone CSVs could be loaded.")

## 2. Averaged Zone Parameters (Final Profile)
These values are the zone-index-averaged plasma parameters stored in the final instrument profile.

In [ ]:
avg_zones_df = pd.read_csv(averaged_zones_csv_path, header=0)
print("Averaged Plasma Zone Parameters:")
print(avg_zones_df[['Te', 'Ne', 'Weight']].to_string(index=False))
print(f"\nAverage R\u00b2  : {rsquare_score:.4f}")
print(f"Average RMSE: {rmse:.4f}")

In [ ]:
# Bar-chart comparison of averaged zone parameters
zone_labels = [f"Zone {i+1}" for i in range(len(avg_zones_df))]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(zone_labels, avg_zones_df['Te'], color='steelblue')
axes[0].set_title('Plasma Temperature (Te)')
axes[0].set_ylabel('Te (eV)')

axes[1].bar(zone_labels, avg_zones_df['Ne'], color='darkorange')
axes[1].set_title('Electron Density (Ne)')
axes[1].set_ylabel('Ne (cm\u207b\u00b3)')

axes[2].bar(zone_labels, avg_zones_df['Weight'], color='seagreen')
axes[2].set_title('Zone Weight')
axes[2].set_ylabel('Weight')

plt.suptitle(f'Averaged Zone Parameters \u2013 {instrument_name}', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Per-Material Synthetic Zone Spectra
Each sub-plot shows the synthetic spectra generated for each plasma zone of a given material.

In [ ]:
for mat_name in material_names:
    csv_path = os.path.join(per_material_zones_dir, f"{mat_name}_best_zones.csv")
    if not os.path.exists(csv_path):
        continue
    mat_df = pd.read_csv(csv_path, header=0)

    # Spectrum data starts at column index 3 (after Te, Ne, Weight)
    spectral_cols = mat_df.columns.to_list()[3:]
    if not spectral_cols:
        continue
    zone_wavelengths = np.array(list(map(float, spectral_cols)))

    # Skip if all-zero (no spectrum data stored)
    if not mat_df.iloc[:, 3:].values.any():
        print(f"  Note: no spectral data stored for '{mat_name}' \u2013 skipping zone spectrum plot.")
        continue

    plt.figure(figsize=(14, 5))
    plt.title(f"Material: {mat_name} \u2013 Plasma Zone Contributions")
    for zone_idx, row in mat_df.iterrows():
        label = f"Zone {zone_idx+1} (Te={row['Te']:.2f} eV, Ne={row['Ne']:.1e} cm\u207b\u00b3)"
        plt.plot(zone_wavelengths, row.iloc[3:].values, label=label, alpha=0.8)
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Intensity (a.u.)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Per-Material Zone Parameter Comparison
Visual comparison of Te and Ne across all processed materials, split by zone.

In [ ]:
if per_mat_rows:
    summary_df = pd.DataFrame(per_mat_rows)
    zones_present = sorted(summary_df['Zone'].unique())

    fig, axes = plt.subplots(len(zones_present), 2, figsize=(14, 5 * len(zones_present)), squeeze=False)

    for row_idx, zone_num in enumerate(zones_present):
        zone_data = summary_df[summary_df['Zone'] == zone_num]
        x_labels = zone_data['Material'].tolist()

        axes[row_idx][0].bar(x_labels, zone_data['Te (eV)'], color='steelblue')
        axes[row_idx][0].set_title(f'Zone {zone_num} \u2013 Te (eV)')
        axes[row_idx][0].set_ylabel('Te (eV)')
        axes[row_idx][0].tick_params(axis='x', rotation=45)

        axes[row_idx][1].bar(x_labels, zone_data['Ne (cm\u207b\u00b3)'], color='darkorange')
        axes[row_idx][1].set_title(f'Zone {zone_num} \u2013 Ne (cm\u207b\u00b3)')
        axes[row_idx][1].set_ylabel('Ne (cm\u207b\u00b3)')
        axes[row_idx][1].tick_params(axis='x', rotation=45)

    plt.suptitle(f'Per-Material Zone Parameters \u2013 {instrument_name}', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No per-material data available for comparison plot.")